In [1]:
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.io as pio
import os
import getpass


In [2]:
# =========================================================
# CARGA DE DATOS
# =========================================================
#TENEMOS DOS RUTAS PARA DOS PROGRAMADORES
RUTA_ALAN = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"
#MIA PERSONAL DE ARCHIVOS
RUTA_ALEX="C:\\Users\\Diego\\Downloads\\proyecto geostat\\dashboard_donatarias-2026\\files"
usuario= getpass.getuser()
if usuario=="Diego":
    RUTA=RUTA_ALEX
else:
    RUTA=RUTA_ALAN
def cargar_datos():
    generales = pd.read_excel(f"{RUTA}/Generales.xlsx")
    donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
    relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
    no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")
    gastos = pd.read_excel(f"{RUTA}/Gastos.xlsx")
    identidad = pd.read_excel(f"{RUTA}/Identidad organizacional.xlsx")
    fl_total_df = pd.read_excel(f"{RUTA}/flota_laboral_total.xlsx", index_col=0)
    sp_df = pd.read_excel(f"{RUTA}/Salario promedio.xlsx", index_col=0)
    dv_df = pd.read_excel(f"{RUTA}/dependencia_voluntariado.xlsx", index_col=0)
    cog_df = pd.read_excel(f"{RUTA}/costo_órgano_gubernamental.xlsx", index_col=0)
    cpb_df = pd.read_excel(f"{RUTA}/costo por beneficiario.xlsx", index_col=0)
    return generales, donativos, relacionados, no_relacionados, gastos, identidad, fl_total_df, sp_df, dv_df, cog_df, cpb_df

generales, donativos, relacionados, no_relacionados, gastos, identidad, fl_total_df, sp_df, dv_df, cog_df, cpb_df = cargar_datos()

LLAVE = ["Año", "Rfc"]  # ajustar 'ID' a la llave real


In [3]:
# =========================================================
# PREPARACIÓN DE INDICADORES BASE 
# =========================================================
don_efectivo = donativos.groupby(LLAVE)["Monto efectivo"].sum().reset_index()
don_especie = donativos.groupby(LLAVE)["Monto especie"].sum().reset_index()
ing_relacionados = relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos relacionados"})
ing_no_relacionados = no_relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos no relacionados"})

total_ingresos = (
    don_efectivo
    .merge(don_especie, on=LLAVE, how="outer")
    .merge(ing_relacionados, on=LLAVE, how="outer")
    .merge(ing_no_relacionados, on=LLAVE, how="outer")
    .fillna(0)
)

total_ingresos["Total Ingresos Anio Fiscal"] = (
    total_ingresos["Monto efectivo"]
    + total_ingresos["Monto especie"]
    + total_ingresos["Ingresos relacionados"]
    + total_ingresos["Ingresos no relacionados"]
)
total_ingresos["Total Donativos Recibidos"] = (
    total_ingresos["Monto efectivo"] + total_ingresos["Monto especie"]
)
total_ingresos["Dependencia de donativos"] = (
    total_ingresos["Total Donativos Recibidos"] / total_ingresos["Total Ingresos Anio Fiscal"]
).fillna(0)

gastos["Total Gastos"] = (
    gastos["Monto nacional operación"] + gastos["Monto nacional admin"]
    + gastos["Monto extranjero operación"] + gastos["Monto extranjero admin"]
)
total_gastos = gastos.groupby(LLAVE)["Total Gastos"].sum().reset_index()

total_ingresos = total_ingresos.merge(total_gastos, on=LLAVE, how="left")
total_ingresos["Total Gastos"] = total_ingresos["Total Gastos"].fillna(0)

total_ingresos["Sustentabilidad"] = (
    (total_ingresos["Total Ingresos Anio Fiscal"] - total_ingresos["Total Gastos"])
    / total_ingresos["Total Ingresos Anio Fiscal"]
)

total_ingresos["Sustentabilidad"] = (
    total_ingresos["Sustentabilidad"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)
print(total_ingresos.columns)
def asignar_quintil(df):
    df = df.copy()
    df["Quintil de ingresos"] = pd.qcut(
        df["Total Ingresos Anio Fiscal"].rank(method="first", ascending=False),
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    return df

total_ingresos = total_ingresos.groupby("Año", group_keys=False).apply(asignar_quintil)
total_ingresos = total_ingresos[total_ingresos["Año"] != 2026]  # sin datos aún

total_ingresos = total_ingresos.merge(
    identidad[["Año", "Rfc", "Identidad organizacional", "Total Beneficiarios", "Costo por beneficiario"]],
    on=["Año", "Rfc"], how="left"
)
total_ingresos["Identidad organizacional"] = total_ingresos["Identidad organizacional"].fillna("No clasificada")


Index(['Año', 'Rfc', 'Monto efectivo', 'Monto especie',
       'Ingresos relacionados', 'Ingresos no relacionados',
       'Total Ingresos Anio Fiscal', 'Total Donativos Recibidos',
       'Dependencia de donativos', 'Total Gastos', 'Sustentabilidad'],
      dtype='object')


C:\Users\Diego\AppData\Local\Temp\ipykernel_3060\3139428702.py:59: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  total_ingresos = total_ingresos.groupby("Año", group_keys=False).apply(asignar_quintil)
